# 奇异值
通过对大模型权重矩阵进行奇异值分解（SVD），研究人员发现强化学习（RL）在参数层面的作用机制与传统的监督微调（SFT）截然不同。
- <font color='red'>简而言之，RL 并没有改变模型内部知识的“重要性”，而是通过微调特定的“方向”来恢复或提升模型的能力。</font>

具体到 SVD 分解后的奇异值与奇异向量，RL 训练后的变化主要体现在以下几个方面：
## 1. 奇异值（Singular Values）：保持惊人的稳定性
<font color='red'>奇异值代表了模型在不同表征方向上的“拉伸强度”或核心知识的重要性。</font>
- 研究表明，无论是 SFT 还是 RL，<font color='red'>微调前后的奇异值谱线几乎完全重叠，变化量接近于零，仅表现为低幅度的随机噪声。</font>
### 结论：
- RL 过程并没有系统性地放大或缩小某些表征模式的“重要性”。<font color='red'>模型的核心“能量”（由奇异值平方和衡量）在 RL 训练中基本保持恒定。</font>

## 2. 奇异向量（Singular Vectors）：发生方向“旋转”
<font color='red'>奇异向量定义了模型内部知识表征的“方向”</font>。与奇异值的稳定不同，<font color='red'>RL 会导致奇异向量方向发生显著变化。</font>
- 恢复机制：
  - 如果模型在 SFT 阶段因为过度拟合而丧失了泛化能力（OOD性能下降）（Out-of-Distribution，分布外性能,是衡量模型在真实世界中是否可靠、是否安全的“终极试金石”。），<font color='red'>RL 的作用更像是一个“修复工”</font>。<font color='blue'>RL 能够将那些在 SFT 中被“错误旋转”的奇异向量方向，部分地“转回去”，从而恢复模型的泛化推理能力。</font>
- 低秩恢复：
  - 实验证明，<font color='red'>仅恢复最重要的前 20% 秩（即最核心的奇异向量方向），就能大幅恢复模型在未见数据上的性能。</font>
  
## 3. RLVR 的“非主成分”更新偏好（三门理论）
在基于可验证奖励的强化学习（RLVR）中，<font color='red'>参数更新表现出高度的稀疏性，并且强烈倾向于避开模型的“主方向”（Principal Directions）：</font>
- 避开“高山”：
  - <font color='red'>权重矩阵中奇异值最大的方向（主成分）储存了模型最核心的基础知识</font>。改变这些方向会引起模型行为的剧烈变化。为了保持模型结构的稳定，<font color='blue'>RL 的优化器会刻意避开这些高曲率的“主方向”</font>。
- 聚焦“平原”：
  - RL 的参数更新主要集中在奇异值较小的方向（非主方向，Off-Principal Directions）。<font color='red'>这些方向对模型整体稳定性的影响较小，修改它们能以较低的代价有效调整模型的解题策略。</font>
  
## 总结而言，
  - 对权重矩阵做 SVD 分析可以清晰地看到：RL 训练保留了模型的奇异值（核心知识的重要性不变），<font color='red'>但通过调整奇异向量的方向（尤其是避开主成分方向，在非主成分空间进行微调），实现了对模型泛化能力的修复与提升。</font>
  
要不要我展开讲讲三门理论（Three Gates）？它从信息论角度解释了RL为什么只更新非主成分方向，挺有意思的。

# “三门理论”（Three Gates Theory）
- 是近年来在研究大语言模型对齐（Alignment）和强化学习（RL）机制时，提出的一种非常深刻的理论框架。
- 它结合了信息论和优化景观（Optimization Landscape）的视角，<font color='red'>完美解释了为什么在 RL 训练（尤其是 RLHF 或 RLVR）中，模型会刻意避开主成分方向，而选择在非主成分方向上进行微调。</font>
- 我们可以把大模型权重矩阵的奇异值谱（Spectrum）想象成一条连绵的山脉，而<font color='red'>“三门理论”将这条山脉划分为了三个功能截然不同的区域</font>（即三道“门”）：

## 第一道门：核心知识门（The Knowledge Gate / 主成分区）
- 位置：
  - 奇异值谱的最前端（Top-k，即最大的奇异值对应的方向）。
- 信息论视角：
  - 这些方向包含了模型在预训练阶段学到的最核心、最通用的世界知识和语言规律。这里的参数包含了极高的信息密度（高熵）。
- RL 中的表现（为什么避开它）：
  - 在数学上，这些方向对应着损失函数曲面上曲率极高的区域（Loss Landscape 上的“高山”）。如果 RL 的梯度更新强行改变这些方向，模型的核心知识就会崩塌，导致严重的“灾难性遗忘”。因此，RL 的优化器（如 PPO 算法）在计算梯度时，会本能地避开这个区域。
  
## 第二道门：行为对齐门（The Alignment Gate / 中间成分区）
- 位置：
  - 奇异值谱的中段（Middle-k）。
- 信息论视角：
  - 这些方向包含了模型的“条件反射”和“表达偏好”。它们不是基础事实，而是模型学会的“如何回答”、“语气是什么”、“遵循什么指令”。
- RL 中的表现（主要修改区）：
  - 这个区域的曲率相对平缓（Loss Landscape 上的“平原”）。RL 的核心目的（如让模型变得有用、无害、遵循特定格式）正好需要修改这些方向。因此，RL 的更新会大量集中在这个区域，通过旋转这些方向的奇异向量，来调整模型的输出风格和安全边界，而不破坏底层的知识。
  
## 第三道门：冗余与噪声门（The Redundancy Gate / 尾部成分区）
- 位置：
  - 奇异值谱的最末端（Bottom-k，即最小的奇异值对应的方向）。
- 信息论视角：
  - 这些方向几乎不包含有效信息，主要是预训练过程中的过拟合噪声、数据偏差或纯粹的参数冗余。
- RL 中的表现（被忽略或用于微调）：
  - 虽然这些方向对模型整体表现影响极小，但有时 RL 也会利用它们。因为改变这些方向几乎不会引起模型行为的任何负面波动。在某些极端的 RLVR（可验证奖励）任务中，模型可能会利用这些空间来“死记硬背”特定的解题套路。
  
## 总结：信息论视角的 RL 本质
“三门理论”从本质上揭示了 RL 与 SFT（监督微调）的区别：
- SFT 像是一个“雕刻家”：
  - 它试图直接重塑整座山脉，既改变知识，又改变风格，因此容易破坏高曲率的核心知识（第一道门）。
- <font color='red'>RL 像是一个“调音师”</font>：
  - 它承认并保护第一道门（核心知识）的绝对稳定，只在第二道门（行为对齐）中进行精细的旋钮调节，偶尔在第三道门（冗余空间）中做微调。

这也解释了为什么我们在 SVD 分析中看到：
- <font color='red'>RL 训练前后，奇异值（能量/重要性）几乎不变，但奇异向量（方向/表达）发生了旋转。</font> <font color='blue'>因为 RL 只是在不改变“知识重量”的前提下，把“表达方向”拨到了正确的位置。</font>

三门理论其实和"线性表示假设"（Linear Representation Hypothesis）有很强的联系，要不要展开讲讲这个框架？

